# Re-run: prediction with all training learners (RQ3)

The `fit` records with `regime: "main"` are each model's five folds with all training learners. This notebook
recomputes Table 5 (mean test AUC over the folds, with the lowest and highest fold), Table 6 (mean log loss),
the fold counts and the seed difference the text cites. It needs only the standard library.

In the per-case outputs, the model the article calls Best-LR with time windows has the identifier `Best-LR+`.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "knowledge-tracing-benchmark-2026"   # set when the report is built
DATA_NAME = 'per-case-outputs.jsonl'     # this release's per-case file, set when the report is built (None: it has none)


def find_release_file(name=None):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    name = name or DATA_NAME
    if not name:
        raise FileNotFoundError(f"Release {RELEASE_ID} publishes no per-case file; this notebook reads its "
                                f"code package (code_dir).")
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        if (d / name).is_file():
            return d / name
        gz = d / (name + ".gz")
        if gz.is_file():
            # The report offers the data file gzipped; unpack it once beside the download, so a notebook that
            # opens the file as text reads the same bytes either way.
            try:
                (d / name).write_bytes(gzip.decompress(gz.read_bytes()))
                return d / name
            except OSError:
                return gz
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def is_this_release(d):
    """True when folder d holds this release, not another one's code/ found higher up: d is named after the
    release, or its code-manifest.txt or a release file (*.yaml) names it."""
    if d.name == RELEASE_ID:
        return True
    texts = [d / "code-manifest.txt", *sorted(d.glob("*.yaml"))]
    return any(f.is_file() and RELEASE_ID in f.read_text(encoding="utf-8", errors="ignore") for f in texts)


def code_dir():
    """The release's code package (its code/ folder), looked for only by a notebook that reads it: in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file() and is_this_release(d):
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-outputs.jsonl


In [2]:
from statistics import fmean
recs = [json.loads(x) for x in open(find_release_file("per-case-outputs.jsonl"), encoding="utf-8")]
fits = [r for r in recs if r["record"] == "fit"]
C = {"half A": "kddcup2010-algebra0607:halfA", "bridge06": "kddcup2010-bridge06", "EDM Cup": "edmcup2023"}
main = {}
for r in fits:
    if r["regime"] == "main":
        main.setdefault((r["model"], r["corpus"]), {})[r["fold"]] = r
f4 = lambda x: f"{x:.4f}"
row = lambda m, c: [main[(m, C[c])][k] for k in range(5)]
auc = lambda m, c: [x["auc"] for x in row(m, c)]

want = {("AKT", "half A"): ("0.8148", "0.8111", "0.8207"), ("simpleKT", "half A"): ("0.8128", "0.8092", "0.8166"),
        ("AKT", "bridge06"): ("0.8149", "0.8133", "0.8165"), ("simpleKT", "bridge06"): ("0.8148", "0.8122", "0.8165"),
        ("AKT", "EDM Cup"): ("0.8381", "0.8354", "0.8423"), ("simpleKT", "EDM Cup"): ("0.8364", "0.8336", "0.8405"),
        ("IRT", "half A"): ("0.7261", "0.7198", "0.7371"), ("Best-LR+", "half A"): ("0.8122", "0.8092", "0.8163"),
        ("HawkesKT", "EDM Cup"): ("0.8035", "0.7997", "0.8074"), ("DKT", "EDM Cup"): ("0.8291", "0.8260", "0.8327")}
for (m, c), w in want.items():
    a = auc(m, c)
    got = (f4(fmean(a)), f4(min(a)), f4(max(a)))
    print(m, c, got)
    assert got == w
for c in C:
    top = sorted({m for m, cc in main if cc == C[c]}, key=lambda m: -fmean(auc(m, c)))[:2]
    assert top == ["AKT", "simpleKT"], (c, top)
print("Highest means: AKT, then simpleKT, on every corpus.")

AKT half A ('0.8148', '0.8111', '0.8207')
simpleKT half A ('0.8128', '0.8092', '0.8166')
AKT bridge06 ('0.8149', '0.8133', '0.8165')
simpleKT bridge06 ('0.8148', '0.8122', '0.8165')
AKT EDM Cup ('0.8381', '0.8354', '0.8423')
simpleKT EDM Cup ('0.8364', '0.8336', '0.8405')
IRT half A ('0.7261', '0.7198', '0.7371')
Best-LR+ half A ('0.8122', '0.8092', '0.8163')
HawkesKT EDM Cup ('0.8035', '0.7997', '0.8074')
DKT EDM Cup ('0.8291', '0.8260', '0.8327')
Highest means: AKT, then simpleKT, on every corpus.


In [3]:
leads = lambda a, b, c: sum(x > y for x, y in zip(auc(a, c), auc(b, c)))
assert [leads("AKT", "simpleKT", c) for c in C] == [4, 2, 5]
assert leads("AKT", "Best-LR+", "bridge06") == 5 and leads("AKT", "Best-LR+", "EDM Cup") == 5
spread = fmean(auc("AKT", "half A")) - fmean(auc("Best-LR+", "half A"))
assert f"{spread:.4f}" == "0.0026"
ll = {(m, c): f"{fmean(x['log_loss'] for x in row(m, c)):.3f}" for (m, cc) in main for c in C if cc == C[c]}
assert ll[("Best-LR+", "half A")] == "0.407" and ll[("AKT", "half A")] == "0.408" and ll[("HawkesKT", "half A")] == "0.664"
assert min((v, m) for (m, c), v in ll.items() if c == "half A")[1] == "Best-LR+"
sat = max(x["saturated_share"] for x in row("HawkesKT", "half A"))
assert f"{sat:.1%}" == "4.0%"
others = [x["saturated_share"] for (m, cc), d in main.items() if m != "HawkesKT" for x in d.values()]
assert max(others) == 0
seed = [abs(r["auc"] - main[(r["model"], r["corpus"])][0]["auc"]) for r in fits if r["regime"] == "seed1"]
assert f"{max(seed):.4f}" == "0.0034"
print("Fold counts, the 0.0026 spread, log loss, saturation (4.0%) and the seed difference (0.0034) match.")

Fold counts, the 0.0026 spread, log loss, saturation (4.0%) and the seed difference (0.0034) match.
